# Proyecto Prediccion de fuga de clientes


In [ ]:

import pandas as pd

print("--- CARGANDO DATASET REAL DE STREAMING ---")

df = pd.read_csv("cuentas streaming(Hoja1).csv")

df.columns = df.columns.str.strip().str.lower()

df = df.loc[:, ~df.columns.str.contains('^unnamed')]

print("Columnas detectadas en el archivo:", df.columns.tolist())
print("-" * 50)

col_cancelacion = [c for c in df.columns if 'cancel' in c]

if col_cancelacion:
    col_exacta = col_cancelacion[0]
    print(f"-> Columna de cancelación detectada exitosamente como: '{col_exacta}'")

    df['churn'] = df[col_exacta].notna().astype(int)
else:
    print("-> ERROR: No se encontró ninguna columna que contenga la palabra 'cancel'.")
    print("Creando columna 'churn' con ceros por defecto para no detener el script.")
    df['churn'] = 0

df.to_csv("dataset_streaming_limpio.csv", index=False)
print("-" * 50)
print("¡Archivo filtrado y guardado como 'dataset_streaming_limpio.csv'!\n")
print("--- INSPECCIÓN REQUERIDA PARA EL AVANCE 1 ---")
print(f"Dimensiones del dataset (Shape): {df.shape}\n")
print("Primeros registros del dataset (Head):")
print(df.head())

--- CARGANDO DATASET REAL DE STREAMING ---
Columnas detectadas en el archivo: ['id_cliente', 'duracion_plan', 'metodo_pago', 'antiguedad_meses', 'horas_uso_semanal', 'dias_desde_ultimo_login', 'reportes_soporte', 'dias_retraso_pago', 'churn']
--------------------------------------------------
-> ERROR: No se encontró ninguna columna que contenga la palabra 'cancel'.
Creando columna 'churn' con ceros por defecto para no detener el script.
--------------------------------------------------
¡Archivo filtrado y guardado como 'dataset_streaming_limpio.csv'!

--- INSPECCIÓN REQUERIDA PARA EL AVANCE 1 ---
Dimensiones del dataset (Shape): (1500, 9)

Primeros registros del dataset (Head):
     id_cliente duracion_plan    metodo_pago  antiguedad_meses  \
0  CLIENTE_0001         1 mes  Transferencia                19   
1  CLIENTE_0002         1 año  Transferencia                 5   
2  CLIENTE_0003         1 año  Transferencia                 8   
3  CLIENTE_0004       3 meses  Transferencia   

In [5]:
from google.colab import drive
import os


drive.mount('/content/drive')

ruta_drive = '/content/drive/MyDrive/'
if not os.path.exists(ruta_drive):
    os.makedirs(ruta_drive)

# OPCIÓN A: Guardar el archivo limpio en Drive para no perderlo
# !cp dataset_streaming_limpio.csv {ruta_drive}

!cp {ruta_drive}'cuentas streaming(Hoja1).csv' /content/

print(f"Directorio en Drive listo: {ruta_drive}")

Mounted at /content/drive
Directorio en Drive listo: /content/drive/MyDrive/


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [ ]:



df = pd.read_csv("cuentas streaming(Hoja1).csv")

# Seleccionar variables predictoras numéricas/categóricas y el target
# Nota: Ajusta las columnas 'X' según las variables finales de tu archivo limpia
X = df[['duracion', 'pago']] # Ejemplo: primero aplica pd.get_dummies si son texto
y = df['churn']

# Si tienes datos categóricos en texto, recuerda convertirlos a números antes:
X = pd.get_dummies(X, drop_first=True)

# 2. División de datos (Como en el Ejemplo 2b de tu práctica)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 3. Creación del Pipeline para evitar Data Leakage (Como en el Ejemplo 3)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(penalty='l2', C=1.0, random_state=42))
    # Puedes probar cambiar a penalty='l1' con solver='liblinear' para ver coeficientes en cero
])

# 4. Entrenamiento
pipe.fit(X_train, y_train)

# 5. Predicciones y Probabilidades (Segunda columna [:, 1] como pide tu guía)
y_pred = pipe.predict(X_test)
y_probs = pipe.predict_proba(X_test)[:, 1]

# 6. Reporte de Métricas exigidas
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred))

print("=== MATRIZ DE CONFUSIÓN ===")
print(confusion_matrix(y_test, y_pred))

# 7. Cálculo de ROC-AUC (Celdas 38-40 de tu práctica)
auc = roc_auc_score(y_test, y_probs)
print(f"\nROC-AUC Score: {auc:.4f}")

# Graficar la curva ROC
fpr, tpr, _ = roc_curve(y_test, y_probs)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, color='red', label=f'Regresión Logística (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='blue', linestyle='--')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC - Predicción de Churn')
plt.legend()
plt.show()